# 🎬 Recomendação de Filmes via Filtragem Colaborativa usando SVD Estável Truncada

Este notebook apresenta a aplicação de Decomposição em Valores Singulares Truncada (`svd_estavel_truncada`) implementada do zero para filtragem colaborativa e recomendação de filmes.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
from src import svd_estavel_truncada

## 1. Criação da Matriz de Avaliação Usuário-Item (Netflix style)
Criamos uma matriz onde `NaN` representa filmes que o usuário ainda não avaliou.

In [ ]:
users = ['Alice', 'Bob', 'Charlie', 'David', 'Eva']
movies = ['Matrix', 'Inception', 'Interstellar', 'Toy Story', 'Finding Nemo', 'Up']

ratings = np.array([
    [5., 4., 5., 1., np.nan, 1.],
    [np.nan, 5., 4., 2., 1., np.nan],
    [1., 1., np.nan, 5., 4., 5.],
    [2., np.nan, 1., 4., 5., 4.],
    [4., 4., np.nan, np.nan, 2., 1.]
])

df_ratings = pd.DataFrame(ratings, index=users, columns=movies)
print('Matriz de avaliações inicial:')
df_ratings

## 2. Imputação de Valores Faltantes por Média
Antes de aplicar SVD, preenchemos os valores faltantes (`NaN`) com a média das avaliações de cada filme.

In [ ]:
col_means = np.nanmean(ratings, axis=0)
ratings_imputed = np.where(np.isnan(ratings), col_means, ratings)

df_imputed = pd.DataFrame(ratings_imputed, index=users, columns=movies)
print('Matriz imputada:')
df_imputed

## 3. Aplicação da SVD Estável Truncada
Decompomos a matriz e geramos a reconstrução estável de baixa ordem ($k=2$ fatores latentes).

In [ ]:
k = 2
U_k, Sigma_k, V_k, A_k = svd_estavel_truncada(ratings_imputed, k)

print('Formato de U_k:', U_k.shape)
print('Valores singulares retidos (Sigma):', np.diag(Sigma_k))
print('Formato de V_k:', V_k.shape)

## 4. Geração de Recomendações personalizadas
A matriz reconstruída $A_k$ fornece as predições de notas para os itens não avaliados.

In [ ]:
df_pred = pd.DataFrame(A_k, index=users, columns=movies)
print('Avaliações preditas (Matriz Reconstruída):')
df_pred

## 5. Extração de Recomendações Práticas
Para cada usuário, identificamos os filmes que ele não assistiu e que têm a maior nota prevista.

In [ ]:
for i, user in enumerate(users):
    unrated_mask = np.isnan(ratings[i])
    user_preds = A_k[i]
    recommended_idx = np.where(unrated_mask)[0]
    
    if len(recommended_idx) > 0:
        sorted_preds = sorted(zip(np.array(movies)[recommended_idx], user_preds[recommended_idx]), key=lambda x: x[1], reverse=True)
        print(f'Recomendações para {user}:')
        for movie, rating in sorted_preds:
            print(f'  - {movie} (Nota prevista: {rating:.2f})')
    else:
        print(f'{user} já avaliou todos os filmes.')